# Assignment 4 — Part 3: Building Blocks of an Energy Model

**Course:** Applied Generative AI  
**Questions:** 6–9  
**Topic:** Gradients and computational graphs in PyTorch

This notebook follows the code and subquestions in the supplied *Assignment 4* PDF. Each answer shows the calculation, a reproducible experiment, and a brief interpretation. Run the cells from top to bottom. All examples run on the CPU and do not require the trained CIFAR-10 models.

## Question 6 — Basic gradient calculations

**Given code:** $x=2$, `requires_grad=True`, and $y=x^2+3x$.

**(a) Expected gradient.** Differentiate term by term:

$$\frac{dy}{dx}=2x+3.$$

At $x=2$, $2(2)+3=\boxed{7}$. After `y.backward()`, we expect `x.grad = tensor([7.])`.

**(b) With `requires_grad=False`.** PyTorch does not build a gradient history for this input when no other participating tensor requires gradients. Calling `y.backward()` raises a `RuntimeError` because `y` does not require gradients. The `x.grad` attribute remains `None`.

**(c) Without specifying the flag.** The default for `torch.tensor` is `requires_grad=False`, so gradients are **not tracked by default**. This matches PyTorch's documented tensor constructor behavior.

In [1]:
import torch

x = torch.tensor([2.0], requires_grad=True)
y = x**2 + 3 * x
y.backward()
print("Q6(a): x.grad =", x.grad)
assert torch.allclose(x.grad, torch.tensor([7.0]))

x_no_grad = torch.tensor([2.0], requires_grad=False)
y_no_grad = x_no_grad**2 + 3 * x_no_grad
print("Q6(b): y.requires_grad =", y_no_grad.requires_grad)
try:
    y_no_grad.backward()
except RuntimeError as exc:
    print("Q6(b): expected RuntimeError:", str(exc).splitlines()[0])
print("Q6(b): x.grad =", x_no_grad.grad)

x_default = torch.tensor([2.0])
print("Q6(c): default requires_grad =", x_default.requires_grad)
assert x_no_grad.grad is None and not x_default.requires_grad

Q6(a): x.grad = tensor([7.])
Q6(b): y.requires_grad = False
Q6(b): expected RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
Q6(b): x.grad = None
Q6(c): default requires_grad = False


## Question 7 — Introduce weights

Let $x=2$ and $w=[1,3]$, with

$$y=w_0x^2+w_1x.$$

**(a)** In the given code, `w = torch.tensor([1.0, 3.0])` does not require gradients. After backpropagation, `w.grad` is therefore **`None`**, even though the derivative with respect to `x` is calculated.

**(b)** Create `w` using `requires_grad=True`. The two weight derivatives are:

$$\frac{\partial y}{\partial w_0}=x^2=4,\qquad \frac{\partial y}{\partial w_1}=x=2.$$

We expect **`w.grad = tensor([4., 2.])`**. The gradient with respect to the input is $2w_0x+w_1=7$.

**(c)** If `requires_grad` is omitted from `torch.tensor`, its default is `False`. A tensor will only have its own gradient stored if gradient tracking is enabled for it (and it is an appropriate leaf tensor).

In [2]:
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([1.0, 3.0])
y = w[0] * x**2 + w[1] * x
y.backward()
print("Q7(a): x.grad =", x.grad)
print("Q7(a): w.grad =", w.grad)
assert w.grad is None

x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([1.0, 3.0], requires_grad=True)
y = w[0] * x**2 + w[1] * x
y.backward()
print("Q7(b): x.grad =", x.grad)
print("Q7(b): w.grad =", w.grad)
assert torch.allclose(w.grad, torch.tensor([4.0, 2.0]))
assert torch.allclose(x.grad, torch.tensor([7.0]))

w_default = torch.tensor([1.0, 3.0])
print("Q7(c): w requires_grad by default =", w_default.requires_grad)

Q7(a): x.grad = tensor([7.])
Q7(a): w.grad = None
Q7(b): x.grad = tensor([7.])
Q7(b): w.grad = tensor([4., 2.])
Q7(c): w requires_grad by default = False


## Question 8 — Breaking the graph

The provided sequence is

```python
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y.detach()
w = z * 2
w.backward()  # raises RuntimeError
```

**Why does it fail?** `detach()` returns a tensor disconnected from the autograd graph that links $y$ to $x$. Thus `z.requires_grad` and `w.requires_grad` are both `False`, and `w.backward()` cannot traverse a graph back to `x`.

**A correct fix that still uses a variable named `z`:** use `z = y.clone()` *instead of* `z = y.detach()`. `clone()` keeps the autograd connection. Then $w=2(3x)=6x$, so $dw/dx=\boxed{6}$.

**Important distinction:** `z = y.detach().requires_grad_(True)` is **not** a valid fix for recovering `x.grad`—it only makes `z` a new leaf tensor. If `z` must literally be the detached tensor, the link to `x` cannot be recovered solely by enabling gradient tracking on `z`; the computation must be rebuilt through `y`.

In [3]:
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y.detach()
w = z * 2
print("Q8 original: z.requires_grad =", z.requires_grad)
try:
    w.backward()
except RuntimeError as exc:
    print("Q8 original: expected RuntimeError:", str(exc).splitlines()[0])

# Correct fix: still uses a variable called z, but do not detach the graph.
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y.clone()
w = z * 2
w.backward()
print("Q8 fixed: x.grad =", x.grad)
assert torch.allclose(x.grad, torch.tensor([6.0]))

# Counterexample: re-enabling gradients on detached z does not restore x.grad.
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y.detach().requires_grad_(True)
(z * 2).backward()
print("Q8 counterexample: z.grad =", z.grad, "; x.grad =", x.grad)
assert x.grad is None

Q8 original: z.requires_grad = False
Q8 original: expected RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
Q8 fixed: x.grad = tensor([6.])
Q8 counterexample: z.grad = tensor([2.]) ; x.grad = None


## Question 9 — Gradient accumulation

The first backward pass differentiates $y_1=2x$, so `x.grad` becomes **2**. The second backward pass differentiates $y_2=3x`, but PyTorch *adds* the new gradient to the existing leaf-tensor gradient, giving **$2+3=5$**.

This behavior is intentional and useful for gradient accumulation over multiple mini-batches. If we want only the current calculation's gradient, reset the existing gradient between backward passes. For example, `x.grad.zero_()` makes the second result **3** (or use `x.grad = None`). During neural-network training, optimizers commonly use `optimizer.zero_grad()` before each independent update.

In [4]:
x = torch.tensor([1.0], requires_grad=True)
y1 = x * 2
y1.backward()
print("Q9 after first backward: x.grad =", x.grad)
assert torch.allclose(x.grad, torch.tensor([2.0]))

y2 = x * 3
y2.backward()
print("Q9 after second backward (accumulated): x.grad =", x.grad)
assert torch.allclose(x.grad, torch.tensor([5.0]))

x.grad.zero_()  # Reset before another independent backward pass.
y3 = x * 3
y3.backward()
print("Q9 after zeroing and backward: x.grad =", x.grad)
assert torch.allclose(x.grad, torch.tensor([3.0]))

Q9 after first backward: x.grad = tensor([2.])
Q9 after second backward (accumulated): x.grad = tensor([5.])
Q9 after zeroing and backward: x.grad = tensor([3.])


---

### Summary

PyTorch tracks derivatives only through operations in the autograd graph. `requires_grad=True` enables gradient tracking for a tensor; `detach()` breaks the connection to upstream tensors; and repeated `backward()` calls accumulate gradients unless we reset them.

**Source:** Supplied *Assignment 4*, Section 3, Questions 6–9.